# 🧠 Baseline B0: Sequential Fine-Tuning (Lower Bound)
### Continual Relation Extraction on FewRel Track A (8 tasks × 10 relations, 5-shot, seed 2021)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/main/notebooks/b0_sequential_ft.ipynb)

Notebook này triển khai thực nghiệm **Baseline B0 (Sequential Fine-Tuning)** làm **Cận dưới (Lower Bound)** khoa học cho bài toán Continual Relation Extraction:
- Huấn luyện tuần tự một cá thể mô hình BERT qua 8 task ($T_1 \to \dots \to T_8$).
- Không có bộ nhớ đệm lưu mẫu cũ ($M=0$, No Replay), không có Regularization.
- Đo lường chân thực hiện tượng **Catastrophic Forgetting** ($AA_8 \approx 8.76\%, BWT = -33.62\%$).


## 1. Kiểm tra GPU Preflight (CUDA Verification)


In [ ]:
import torch

print('=' * 60)
print('KIỂM TRA THIẾT BỊ TÍNH TOÁN (GPU PREFLIGHT)')
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ CUDA Khả dụng: True')
    print(f'✓ GPU Thiết bị: {torch.cuda.get_device_name(0)}')
    print(f'✓ Bộ nhớ VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    !nvidia-smi
else:
    print('⚠️ CẢNH BÁO: CUDA KHÔNG KHẢ DỤNG! Bạn đang chạy trên CPU.')
    print('👉 Hướng dẫn bật GPU: Chọn Runtime trên thanh menu -> Change runtime type -> Chọn T4 GPU -> Lưu.')


## 2. Thiết lập Codebase & Cài đặt Thư viện


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
REPO_NAME = 'research-code-lab'

if not Path(REPO_NAME).exists():
    print(f'Đang clone repository từ {REPO_URL}...')
    !git clone {REPO_URL}

%cd {REPO_NAME}

# Cài đặt các thư viện cần thiết
!pip install -q transformers torch matplotlib pandas


## 3. Tự động tải Dữ liệu FewRel thô (Zero-Manual Setup)


In [ ]:
import urllib.request
from pathlib import Path

data_dir = Path('dataset-pipelines/continual-relation-extraction/data/raw/fewrel')
data_dir.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/thunlp/FewRel/master/data'
FILES = ['train_wiki.json', 'val_wiki.json', 'pid2name.json']

for fname in FILES:
    fpath = data_dir / fname
    if not fpath.exists():
        print(f'Đang tải {fname} từ {BASE_URL}/{fname}...')
        urllib.request.urlretrieve(f'{BASE_URL}/{fname}', fpath)
        print(f'  ✓ Hoàn tất {fname} ({fpath.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'  ✓ Đã có sẵn {fname} ({fpath.stat().st_size / 1024:.1f} KB)')

print('Dữ liệu FewRel đã sẵn sàng 100%!')


## 4. Thực thi Huấn luyện Tuần tự Baseline B0 (T1 → T8)


In [ ]:
# Chạy toàn bộ tiến trình Sequential Fine-Tuning
# Mô hình sẽ tự động dùng device CUDA nếu có GPU
!python experiments/run_sequential_ft.py --device auto


## 5. Kiểm chứng Nghiệm thu Tự động (Definition of Done)


In [ ]:
# Kiểm tra 8 chốt chặn DoD và cập nhật đồ thị vector chuẩn xuất bản
!python experiments/validate_b0_results.py
!python experiments/generate_plots_svg.py


## 6. Trực quan hóa Kết quả & Đường cong Lãng quên tại chỗ


In [ ]:
import json
from IPython.display import Image, display, Markdown

results_dir = Path('results/fewrel/5shot/B0_sequential_ft/seed_2021')
summary_path = results_dir / 'summary.json'

if summary_path.exists():
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    print('=' * 60)
    print('B0 SEQUENTIAL FINE-TUNING — KẾT QUẢ THỰC NGHIỆM')
    print('=' * 60)
    print(f'Final Average Accuracy (Final AA): {summary["final_average_accuracy"] * 100:.2f}%')
    print(f'Final Macro-F1:                    {summary.get("final_macro_f1", 0.0) * 100:.2f}%')
    print(f'Average Incremental Accuracy (AIA):{summary["average_incremental_accuracy"] * 100:.2f}%')
    print(f'Average Catastrophic Forgetting:   {summary["average_forgetting"] * 100:.2f}%')
    print(f'Backward Transfer (BWT):           {summary["backward_transfer"] * 100:.2f}%')
    most_f = summary.get('most_forgotten_task', {})
    print(f'Most Forgotten Task:               {most_f.get("task_name")} (drop: {float(most_f.get("forgetting", 0.0)) * 100:.2f}%)')
    print('=' * 60)

forgetting_plot = results_dir / 'plots/forgetting_curve.png'
acc_plot = results_dir / 'plots/accuracy_over_tasks.png'

if forgetting_plot.exists():
    print('\n📊 Biểu đồ Đường cong Lãng quên (Forgetting Curves):')
    display(Image(str(forgetting_plot), width=900))

if acc_plot.exists():
    print('\n📊 Biểu đồ Tiến hóa Độ chính xác (Accuracy Evolution):')
    display(Image(str(acc_plot), width=900))


## 7. Tải Artifacts về Máy tính Cá nhân (Optional)


In [ ]:
import shutil

try:
    from google.colab import files
    zip_name = 'B0_sequential_ft_results'
    print('Đang nén kết quả thành file zip...')
    shutil.make_archive(zip_name, 'zip', results_dir)
    files.download(f'{zip_name}.zip')
    print(f'✓ Đã bắt đầu tải xuống {zip_name}.zip')
except ImportError:
    print('Chức năng tải file tự động chỉ hoạt động trên Google Colab.')
